In [ ]:
# Preparação para o Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Altere o caminho caso o nome da pasta no seu Drive seja diferente
    caminho_pasta = '/content/drive/MyDrive/artefatos colab'
    if os.path.exists(caminho_pasta):
        os.chdir(caminho_pasta)
        print('Diretório alterado para:', os.getcwd())
    else:
        print('ATENÇÃO: Pasta não encontrada no Drive. Verifique se o nome está correto.')
except ImportError:
    print('Não está rodando no Google Colab. Mantendo diretório atual.')


# Sprint 4 — Pipeline RAG e Assistente Conversacional
Neste notebook construímos o RAG sobre a documentação técnica (Sprints 1 e 2) e instanciamos o Assistente Conversacional (LLM).

In [ ]:
!pip install sentence-transformers faiss-cpu langchain langchain-groq langchain-text-splitters langchain-openai -q


## 1. Chunking e Indexação (FAISS)

In [ ]:
corpus_textos = [
    """# Manual do Motor W22 Plus\n\n## 1. Lubrificação e Manutenção\nOs rolamentos devem ser lubrificados a cada 2.000 horas de operacao ou 6 meses. O torque de aperto dos parafusos de fixacao deve ser de 25 N.m.\n\n## 2. Limites Operacionais\nA vibracao maxima permitida eh de 2,8 mm/s RMS conforme ISO 10816. Corrente de partida (Ia/In) 6,5x a corrente nominal.\n""",
    """# Siemens 1LA7 Series Manual\n\n## 1. Commissioning\nInsulation resistance must be measured before commissioning. Minimum insulation resistance 100 MOhm at 1000V DC 60 seconds.\n\n## 2. Maintenance and Limits\nRegreasing interval 3500 hours for bearings under normal load. Vibration limits per ISO 10816 Class B less than 2.8 mm/s RMS. Temperatura maxima do enrolamento 155C Classe F.\n"""
]

# 1. Chunking Semântico com Markdown
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

docs_markdown = []
for text in corpus_textos:
    docs_markdown.extend(markdown_splitter.split_text(text))

# 2. Chunking Secundário Recursivo
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = text_splitter.split_documents(docs_markdown)

chunk_texts = [chunk.page_content for chunk in chunks]
chunk_metadata = [chunk.metadata for chunk in chunks]

print(f"Total de {len(chunk_texts)} chunks gerados.")

# 3. Indexação no FAISS
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(chunk_texts, convert_to_numpy=True, normalize_embeddings=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"Indexado {index.ntotal} vetores no FAISS.")

## 2. LLM e Integração de Contexto (Assistente RAG)

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 1. Configurando a chave de Fallback do OpenRouter
os.environ["OPENROUTER_API_KEY"] = ""
chat_history = []

template = """Você é um Assistente Técnico Especialista em Motores Elétricos.
Sua função é auxiliar operadores no diagnóstico de falhas, sempre baseando-se RIGOROSAMENTE nos manuais fornecidos.

[ESTADO ATUAL DO ATIVO (TELEMETRIA/ALERTAS)]
{alerta_atual}

[MANUAIS RECUPERADOS]
{contexto}

[HISTÓRICO DA CONVERSA]
{chat_history}

[NOVA PERGUNTA]
{pergunta}

INSTRUÇÕES:
1. Responda à pergunta baseando-se APENAS nos [MANUAIS RECUPERADOS].
2. Se a informação não estiver lá, diga explicitamente: "Não possuo informações suficientes na documentação técnica recuperada".
3. Leve em consideração o [ESTADO ATUAL DO ATIVO] para contextualizar a gravidade da situação.
4. Ao final da resposta, classifique seu "Nível de Confiança" (ALTO, MÉDIO, BAIXO) e cite as fontes.

Resposta:"""

prompt_template = PromptTemplate(
    input_variables=["alerta_atual", "contexto", "chat_history", "pergunta"],
    template=template
)

try:
    # 2. Inicializando via LangChain OpenAI direcionando para o OpenRouter
    llm = ChatOpenAI(
        model="nvidia/nemotron-3.5-lightning:free",
        openai_api_key=os.environ["OPENROUTER_API_KEY"],
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.0
    )
    chain = prompt_template | llm
except Exception as e:
    print("Aviso: Falha ao inicializar LLM, verifique a API KEY.", e)
    llm = None
    chain = None

def chat_troubleshooting(user_input, alerta_ativo):
    contexto_recuperado, _ = buscar_contexto_com_rerank(user_input, alerta_ativo, top_k=2)
    historico_str = "\n".join(chat_history) if chat_history else "Nenhum histórico"
    
    print("\n=== PROMPT ENVIADO AO LLM ===")
    print(prompt_template.format(
        alerta_atual=alerta_ativo,
        contexto=contexto_recuperado,
        chat_history=historico_str,
        pergunta=user_input
    ))
    print("===============================\n")

    if chain:
        try:
            resposta = chain.invoke({
                "alerta_atual": alerta_ativo, 
                "contexto": contexto_recuperado, 
                "chat_history": historico_str, 
                "pergunta": user_input
            })
            resposta_llm = resposta.content
        except Exception as e:
            resposta_llm = f"Erro na inferência do LLM: {e}"
    else:
        resposta_llm = "[ERRO] Configuração do LLM ausente."

    chat_history.append(f"Usuário: {user_input}")
    chat_history.append(f"Assistente: {resposta_llm}")

    return resposta_llm


In [ ]:
import json

print("## Carregando Ground Truth (dados_avaliacao.json)")
try:
    with open('dados_avaliacao.json', 'r', encoding='utf-8') as f:
        dados = json.load(f)
    qa_list = dados['qa_troubleshooting']
    print(f"Carregadas {len(qa_list)} perguntas de teste.")
except Exception as e:
    print("Dataset não encontrado no caminho padrão. Usando mock gerado...", e)
    qa_list = [{"pergunta": "Qual a temperatura máxima permitida para o enrolamento do Siemens 1LA7?", "ground_truth": "A temperatura máxima do enrolamento é de 155°C (Classe F)."}]

def calcular_metricas_rag(pergunta, ground_truth, resposta_llm, contexto):
    if not llm:
        return 0.0, 0.0, 0.0

    prompt_faithfulness = f"""
    Avalie se a seguinte resposta baseia-se EXCLUSIVAMENTE no contexto fornecido.
    Contexto: {contexto}
    Resposta: {resposta_llm}
    A resposta é fiel ao contexto (nao inventa fatos)? Responda apenas "SIM" ou "NAO".
    """
    try:
        resultado_f = llm.invoke(prompt_faithfulness).content
        faithfulness = 1.0 if "SIM" in resultado_f.upper() else 0.0
    except:
        faithfulness = 0.0

    prompt_relevancy = f"""
    Avalie se a resposta atende à pergunta original, sendo util e direta.
    Pergunta: {pergunta}
    Resposta: {resposta_llm}
    A resposta é relevante para a pergunta? Responda apenas "SIM" ou "NAO".
    """
    try:
        resultado_ar = llm.invoke(prompt_relevancy).content
        answer_relevancy = 1.0 if "SIM" in resultado_ar.upper() else 0.0
    except:
        answer_relevancy = 0.0

    context_precision = 1.0 if ground_truth[:10].lower() in contexto.lower() else 0.5

    return context_precision, faithfulness, answer_relevancy

print("\n## Rodando Avaliação RAGAS via LLM-as-a-judge (Simulada)...\n")
scores = {"context_precision": [], "faithfulness": [], "answer_relevancy": []}

for qa in qa_list:
    p = qa['pergunta']
    gt = qa['ground_truth']

    alerta_neutro = "Estado Operacional Normal"
    contexto, top_res = buscar_contexto_com_rerank(p, alerta_neutro, top_k=2)

    historico = "\n".join(chat_history) if 'chat_history' in globals() and chat_history else ""
    
    if chain:
        try:
            resp_obj = chain.invoke({
                "alerta_atual": alerta_neutro, 
                "contexto": contexto, 
                "chat_history": historico, 
                "pergunta": p
            })
            resp = resp_obj.content
        except Exception:
            resp = "Erro"
    else:
        resp = "Simulação sem LLM"

    cp, f, ar = calcular_metricas_rag(p, gt, resp, contexto)
    scores["context_precision"].append(cp)
    scores["faithfulness"].append(f)
    scores["answer_relevancy"].append(ar)

media_cp = sum(scores["context_precision"]) / len(qa_list) if len(qa_list) > 0 else 0
media_f = sum(scores["faithfulness"]) / len(qa_list) if len(qa_list) > 0 else 0
media_ar = sum(scores["answer_relevancy"]) / len(qa_list) if len(qa_list) > 0 else 0

print("===" * 15)
print("🏆 RESULTADO FINAL DA AVALIAÇÃO RAG (LLM Judge)")
print("===" * 15)
print(f"-> Context Precision : {media_cp:.2f}")
print(f"-> Faithfulness      : {media_f:.2f}")
print(f"-> Answer Relevancy  : {media_ar:.2f}")
print("===" * 15)


## Limites do Sistema e Cenários de Falha Documentados

### Cenários de Falha Validados
1. **Anomalia Elétrica:** Quando a telemetria indica pico de corrente (ex: Corrente atingiu 7x In), o RAG resgata e o LLM alerta que o limite de partida no manual W22 é 6,5x, caracterizando falha.
2. **Anomalia Mecânica:** Vibração em 2,9 mm/s. O re-ranking garante que a norma ISO 10816 (limite 2,8) venha no topo para os motores listados.
3. **Consulta Preventiva:** Operador solicita periodicidade de lubrificação sem alerta ativo. O FAISS recupera corretamente as 2000 ou 3500 horas, dependendo do motor.

### Limites e Restrições (Tratamento de Alucinação)
* **Out-of-Scope (OOS):** Caso o operador faça perguntas fora dos manuais carregados (ex: "Qual a pressão da bomba hidráulica 02?"), o `PromptTemplate` instrui o modelo a realizar o fallback fixo: *"Não possuo informações suficientes na documentação técnica recuperada"*. Isso garante `Faithfulness = 1.0` (sem alucinação).
* **Restrição de Telemetria:** O re-ranking falha se o nome do ativo reportado pela telemetria não bater de forma exata com as chaves extraídas no *MarkdownHeaderTextSplitter* (Ex: "Motor 1" em vez de "Siemens 1LA7").

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 1. Configurando a chave de Fallback do OpenRouter
os.environ["OPENROUTER_API_KEY"] = ""
chat_history = []

template = """Você é um Assistente Técnico Especialista em Motores Elétricos.
Sua função é auxiliar operadores no diagnóstico de falhas, sempre baseando-se RIGOROSAMENTE nos manuais fornecidos.

[ESTADO ATUAL DO ATIVO (TELEMETRIA/ALERTAS)]
{alerta_atual}

[MANUAIS RECUPERADOS]
{contexto}

[HISTÓRICO DA CONVERSA]
{chat_history}

[NOVA PERGUNTA]
{pergunta}

INSTRUÇÕES:
1. Responda à pergunta baseando-se APENAS nos [MANUAIS RECUPERADOS].
2. Se a informação não estiver lá, diga explicitamente: "Não possuo informações suficientes na documentação técnica recuperada".
3. Leve em consideração o [ESTADO ATUAL DO ATIVO] para contextualizar a gravidade da situação.
4. Ao final da resposta, classifique seu "Nível de Confiança" (ALTO, MÉDIO, BAIXO) e cite as fontes.

Resposta:"""

prompt_template = PromptTemplate(
    input_variables=["alerta_atual", "contexto", "chat_history", "pergunta"],
    template=template
)

try:
    # 2. Inicializando via LangChain OpenAI direcionando para o OpenRouter
    llm = ChatOpenAI(
        model="nvidia/nemotron-3.5-lightning:free",
        openai_api_key=os.environ["OPENROUTER_API_KEY"],
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.0
    )
    chain = prompt_template | llm
except Exception as e:
    print("Aviso: Falha ao inicializar LLM, verifique a API KEY.", e)
    llm = None
    chain = None

def chat_troubleshooting(user_input, alerta_ativo):
    contexto_recuperado, _ = buscar_contexto_com_rerank(user_input, alerta_ativo, top_k=2)
    historico_str = "\n".join(chat_history) if chat_history else "Nenhum histórico"
    
    print("\n=== PROMPT ENVIADO AO LLM ===")
    print(prompt_template.format(
        alerta_atual=alerta_ativo,
        contexto=contexto_recuperado,
        chat_history=historico_str,
        pergunta=user_input
    ))
    print("===============================\n")

    if chain:
        try:
            resposta = chain.invoke({
                "alerta_atual": alerta_ativo, 
                "contexto": contexto_recuperado, 
                "chat_history": historico_str, 
                "pergunta": user_input
            })
            resposta_llm = resposta.content
        except Exception as e:
            resposta_llm = f"Erro na inferência do LLM: {e}"
    else:
        resposta_llm = "[ERRO] Configuração do LLM ausente."

    chat_history.append(f"Usuário: {user_input}")
    chat_history.append(f"Assistente: {resposta_llm}")

    return resposta_llm


In [ ]:
# ==========================================
print("🚀 SISTEMA DE TROUBLESHOOTING INICIADO")
# ==========================================
# Instruções:
# 1. Execute esta célula.
# 2. Digite suas perguntas na caixa de texto.
# 3. Digite 'sair' para encerrar a simulação.
# ==========================================

alerta_ativo = "ALERTA CRÍTICO: Motor W22 Plus - Temperatura atingiu 47°C e vibração 0.47g."

print("==========================================")
print(f"📡 TELEMETRIA ATIVA: {alerta_ativo}")
print("==========================================")
print("Dica de teste: Pergunte sobre os limites de vibração ou temperatura.")

# Descomente o bloco abaixo para usar no Jupyter/Colab de forma interativa:
'''
while True:
    user_input = input("\n👤 Operador: ")
    if user_input.lower() in ['sair', 'exit', 'quit']:
        print("🔌 Encerrando sistema...")
        break

    resposta = chat_troubleshooting(user_input, alerta_ativo)
    print(f"\n🤖 Assistente: {resposta}")
'''
# Apenas rodando um teste fixo para não travar a execução headless
print("\n👤 Operador (mock): Quais os limites de vibração?")
resposta_mock = chat_troubleshooting('Quais os limites de vibração?', alerta_ativo)
print(f"🤖 Assistente: {resposta_mock}\n")
